# R1b: every item's EN and FR names, and the Wikipedia redirects (order 0046)

**What this fetches, and only this** (the claims are never read):
- Wikidata's labels (`en`, `fr`, `mul`), aliases (`en`, `fr`) and enwiki / frwiki titles, from the `wikidata-pq` tables
  on the Hugging Face Hub (CC0), release **20260928** (Wikidata's weekly dump of 2026-09-28). The tables are that dump
  split by language and site, so only 8 files are read: **1.36 GB, 119.4M names, an extract of ~1.5 GB**.
- The EN and FR Wikipedia `page` and `redirect` tables (dumps.wikimedia.org, gzipped SQL, **~3.1 GB streamed**), kept
  as article-space redirect pairs only.

Nothing raw is stored: each file is streamed and only the extract lands on Drive, the master copy
(`cubbyllm/r1b_names/`). **CPU only:** pick a CPU runtime (no GPU units spent).

**The stop:** cell 3 prints the plan (bytes per file, the extract's size) and writes it to Drive. Cell 4 runs only
when `GO = True`, after Nick reads the plan. (The plan above was measured from home on 2026-10-31; this cell measures
it again where it runs.)

In [ ]:
!git clone -q https://github.com/Grillcheese-AI/CubbyLLM.git /content/CubbyLLM 2>/dev/null || git -C /content/CubbyLLM pull -q
!pip -q install -U pyarrow huggingface_hub
from google.colab import drive; drive.mount('/content/drive')
import os, subprocess, sys
REPO = '/content/CubbyLLM'
OUT = '/content/drive/MyDrive/cubbyllm/r1b_names'
os.makedirs(OUT, exist_ok=True)
for f in ('validation/retrieval_sets/names_extract.py', 'validation/retrieval_sets/wiki_redirects.py'):
    assert os.path.exists(f'{REPO}/{f}'), f'push {f} first'


def run(*args):
    # stream the script's output as it comes (a captured run shows nothing until it ends)
    p = subprocess.Popen([sys.executable, '-u', *args], cwd=REPO, text=True, bufsize=1,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end='')
    assert p.wait() == 0, f'{args[0]} failed ({p.returncode})'

## 1. The plan (reads the files' footers and first row groups, and the dumps' headers)

In [ ]:
run('validation/retrieval_sets/names_extract.py', 'plan', '--out', OUT)
run('validation/retrieval_sets/wiki_redirects.py', 'plan', '--out', OUT)

## 2. The extract (after Nick's go on the plan)

Resumable: a file already on Drive is skipped, so re-running after a disconnect continues where it stopped.

In [ ]:
GO = False          # Nick: set True after reading the plan above
assert GO, 'read the plan first, then set GO = True'
run('validation/retrieval_sets/names_extract.py', 'names', '--out', OUT)
run('validation/retrieval_sets/wiki_redirects.py', 'pairs', '--out', OUT)
!du -sh "$OUT"/* 2>/dev/null

## 3. Done

The extract is on Drive (`cubbyllm/r1b_names/names_20260928/`, `redirects_enwiki.parquet`, `redirects_frwiki.parquet`,
the manifests). **Disconnect the runtime.** Claude Code builds the VM segment from it at home (names and aliases to
items, `cubelang knowledge build`), copies it to the SSD, and scores R1b against the lines written in the order.